In [32]:
import numpy as np

In [33]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
from pathlib import Path
import pandas as pd
import numpy as np
import cv2
import json
import os
import time

from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download, login

# Project paths
PROJECT_DIR = Path("/content/drive/MyDrive/Nexar_Project")

METADATA_DIR = PROJECT_DIR / "Metadata"
OUTPUT_DIR = PROJECT_DIR / "Dataset_2FPS"

POSITIVE_DIR = OUTPUT_DIR / "positive"
NEGATIVE_DIR = OUTPUT_DIR / "negative"

MANIFEST_DIR = PROJECT_DIR / "Manifests"
LOG_DIR = PROJECT_DIR / "Logs"

for folder in [
    POSITIVE_DIR, NEGATIVE_DIR,
    MANIFEST_DIR, LOG_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

# Sampling settings
FPS_SAMPLE = 2
IMAGE_SIZE = (224, 224)

REPO_ID = "nexar-ai/nexar_collision_prediction"

# Load metadata
metadata_df = pd.read_csv(METADATA_DIR / "all_metadata.csv")

print("Project is ready.")
print("Total videos:", len(metadata_df))
print("Positive:", (metadata_df["label"] == 1).sum())
print("Negative:", (metadata_df["label"] == 0).sum())
print("Sampling rate:", FPS_SAMPLE, "FPS")

Project is ready.
Total videos: 1500
Positive: 750
Negative: 750
Sampling rate: 2 FPS


In [35]:
login()

In [36]:
def process_video(row):

    label = int(row["label"])
    filename = str(row["file_name"])

    category = "positive" if label == 1 else "negative"

    output_dir = POSITIVE_DIR if label == 1 else NEGATIVE_DIR
    output_path = output_dir / f"{Path(filename).stem}.npz"

    # Resume: skip already processed videos
    if output_path.exists():
        return "skipped", 0

    # Download video to Colab's temporary storage
    video_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=f"train/{category}/{filename}",
        repo_type="dataset"
    )

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise RuntimeError("Cannot open video")

    video_fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if video_fps <= 0 or total_frames <= 0:
        cap.release()
        raise RuntimeError("Invalid video properties")

    duration = total_frames / video_fps

    # Positive: strictly before collision
    # Negative: entire video
    if label == 1:
        event_time = float(row["time_of_event"])
        end_time = min(event_time, duration)
    else:
        end_time = duration

    frames = []
    timestamps = []

    for timestamp in np.arange(0, end_time, 1 / FPS_SAMPLE):

        cap.set(cv2.CAP_PROP_POS_MSEC, timestamp * 1000)
        success, frame = cap.read()

        if not success:
            continue

        frame = cv2.resize(frame, IMAGE_SIZE)
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        frames.append(frame)
        timestamps.append(timestamp)

    cap.release()

    if not frames:
        raise RuntimeError("No frames extracted")

    # Save locally first, then move complete file to Drive
    temp_path = output_path.with_name(output_path.stem + ".tmp.npz")

    np.savez_compressed(
        temp_path,
        frames=np.asarray(frames, dtype=np.uint8),
        timestamps=np.asarray(timestamps, dtype=np.float32)
    )

    os.replace(temp_path, output_path)

    return "success", len(frames)

In [37]:

import os
import cv2
import uuid
import shutil
import logging
import numpy as np
import pandas as pd

from pathlib import Path
from concurrent.futures import (
    ThreadPoolExecutor,
    wait,
    FIRST_COMPLETED
)
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm


# =========================================================
# 1. Configuration
# =========================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Nexar_Project")

METADATA_PATH = PROJECT_DIR / "Metadata/all_metadata.csv"

DRIVE_NEGATIVE = PROJECT_DIR / "Dataset_2FPS/negative"
STAGING_NEGATIVE = Path("/content/Nexar_Staging/negative")

LOG_DIR = PROJECT_DIR / "Logs"

HF_REPO_ID = "nexar-ai/nexar_collision_prediction"

FPS_SAMPLE = 2
IMAGE_SIZE = (224, 224)

MAX_WORKERS = 3
BATCH_SIZE = 20

# Limit queued tasks to reduce memory and resource pressure
MAX_PENDING_TASKS = MAX_WORKERS * 2

for folder in [
    DRIVE_NEGATIVE,
    STAGING_NEGATIVE,
    LOG_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

# Avoid CPU thread oversubscription
cv2.setNumThreads(1)

logging.basicConfig(
    filename=str(LOG_DIR / "negative_processing.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)


# =========================================================
# 2. Validate NPZ files
# =========================================================

def is_valid_npz(path):

    path = Path(path)

    if not path.exists() or path.stat().st_size == 0:
        return False

    try:
        with np.load(path, allow_pickle=False) as data:

            if "frames" not in data or "timestamps" not in data:
                return False

            frames = data["frames"]
            timestamps = data["timestamps"]

            if len(frames) == 0:
                return False

            if len(frames) != len(timestamps):
                return False

            if frames.ndim != 4:
                return False

            if frames.shape[1:] != (224, 224, 3):
                return False

            if frames.dtype != np.uint8:
                return False

            return True

    except Exception:
        return False


# =========================================================
# 3. Load metadata and identify missing videos
# =========================================================

df = pd.read_csv(METADATA_PATH)

df["label"] = pd.to_numeric(
    df["label"],
    errors="coerce"
)

negative_df = df[df["label"] == 0].copy()

# Avoid duplicate processing jobs
negative_df = negative_df.drop_duplicates(
    subset=["file_name"]
)

jobs = []

for _, row in negative_df.iterrows():

    filename = Path(str(row["file_name"])).name

    drive_output = DRIVE_NEGATIVE / (
        Path(filename).stem + ".npz"
    )

    if not is_valid_npz(drive_output):
        jobs.append(row.to_dict())


print(f"Total negative videos: {len(negative_df)}")
print(f"Already completed: {len(negative_df) - len(jobs)}")
print(f"Remaining videos: {len(jobs)}")


# =========================================================
# 4. Download and process one video
# =========================================================

def process_video(row):

    filename = Path(str(row["file_name"])).name
    stem = Path(filename).stem

    local_output = STAGING_NEGATIVE / f"{stem}.npz"

    video_path = None
    cap = None
    temp_path = None

    try:

        # Reuse an existing valid local output
        if is_valid_npz(local_output):

            return {
                "status": "success",
                "filename": filename,
                "local_path": str(local_output)
            }

        # Download video from Hugging Face
        # Existing cached files are reused automatically
        video_path = hf_hub_download(
            repo_id=HF_REPO_ID,
            filename=f"train/negative/{filename}",
            repo_type="dataset"
        )

        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            raise RuntimeError("Could not open video.")

        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(
            cap.get(cv2.CAP_PROP_FRAME_COUNT)
        )

        if fps <= 0 or frame_count <= 0:
            raise RuntimeError("Invalid video properties.")

        duration = frame_count / fps

        # Sample the entire negative video at 2 FPS
        timestamps = np.arange(
            0,
            duration,
            1.0 / FPS_SAMPLE
        )

        frame_indices = np.unique(
            np.round(timestamps * fps).astype(int)
        )

        frame_indices = frame_indices[
            frame_indices < frame_count
        ]

        frames = []
        saved_timestamps = []

        current_frame = 0

        # Sequential decoding: skip unnecessary frames
        for target_idx in frame_indices:

            while current_frame <= target_idx:

                if not cap.grab():
                    break

                current_frame += 1

            if current_frame <= target_idx:
                break

            success, frame = cap.retrieve()

            if not success:
                continue

            frame = cv2.resize(
                frame,
                IMAGE_SIZE,
                interpolation=cv2.INTER_AREA
            )

            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            frames.append(frame)
            saved_timestamps.append(target_idx / fps)

        if not frames:
            raise RuntimeError("No frames extracted.")

        frames = np.asarray(
            frames,
            dtype=np.uint8
        )

        saved_timestamps = np.asarray(
            saved_timestamps,
            dtype=np.float32
        )

        # Unique temporary filename for each worker
        temp_path = STAGING_NEGATIVE / (
            f"{stem}_{uuid.uuid4().hex}.tmp.npz"
        )

        np.savez_compressed(
            temp_path,
            frames=frames,
            timestamps=saved_timestamps
        )

        if not is_valid_npz(temp_path):
            raise RuntimeError("Temporary NPZ validation failed.")

        # Atomic local save
        os.replace(temp_path, local_output)

        return {
            "status": "success",
            "filename": filename,
            "local_path": str(local_output)
        }

    except Exception as e:

        logging.exception(f"Failed: {filename}")

        return {
            "status": "failed",
            "filename": filename,
            "error": str(e)
        }

    finally:

        if cap is not None:
            cap.release()

        if temp_path is not None and temp_path.exists():
            temp_path.unlink()


# =========================================================
# 5. Transfer a batch to Google Drive
# =========================================================

def transfer_batch(results):

    transferred = 0
    failed = 0

    for result in results:

        if result["status"] != "success":
            continue

        local_path = Path(result["local_path"])

        drive_path = DRIVE_NEGATIVE / local_path.name

        # Do not overwrite an already valid Drive file
        if is_valid_npz(drive_path):

            if local_path.exists():
                local_path.unlink()

            continue

        if not is_valid_npz(local_path):
            failed += 1
            continue

        # Unique temporary Drive filename
        temp_drive_path = DRIVE_NEGATIVE / (
            f"{drive_path.stem}_{uuid.uuid4().hex}.uploading.npz"
        )

        try:

            shutil.copy2(
                local_path,
                temp_drive_path
            )

            if not is_valid_npz(temp_drive_path):
                raise RuntimeError("Drive copy validation failed.")

            # Atomic replacement after successful validation
            os.replace(
                temp_drive_path,
                drive_path
            )

            # Remove local staging copy only after successful transfer
            local_path.unlink(missing_ok=True)

            transferred += 1

        except Exception as e:

            logging.exception(
                f"Transfer failed: {drive_path.name}"
            )

            failed += 1

            if temp_drive_path.exists():
                temp_drive_path.unlink()

            print(
                f"Transfer error: {drive_path.name} | {e}"
            )

    return transferred, failed


# =========================================================
# 6. Process videos with controlled parallel execution
# =========================================================

success_count = 0
failed_count = 0
transfer_count = 0
transfer_failed_count = 0

pending_results = []
jobs_iterator = iter(jobs)

if jobs:

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        active_futures = {}

        # Start only a limited number of tasks
        for _ in range(
            min(MAX_PENDING_TASKS, len(jobs))
        ):

            row = next(jobs_iterator, None)

            if row is not None:
                future = executor.submit(
                    process_video,
                    row
                )
                active_futures[future] = True

        progress = tqdm(
            total=len(jobs),
            desc="Processing negative videos"
        )

        while active_futures:

            completed, _ = wait(
                active_futures,
                return_when=FIRST_COMPLETED
            )

            for future in completed:

                active_futures.pop(future)

                result = future.result()
                progress.update(1)

                if result["status"] == "success":

                    success_count += 1
                    pending_results.append(result)

                else:

                    failed_count += 1

                    print(
                        f"Failed: {result['filename']} | "
                        f"{result['error']}"
                    )

                # Transfer each batch of 20 successful videos
                if len(pending_results) >= BATCH_SIZE:

                    transferred, transfer_failed = transfer_batch(
                        pending_results
                    )

                    transfer_count += transferred
                    transfer_failed_count += transfer_failed

                    print(
                        f"\nBatch transferred: {transferred}"
                        f" | Transfer errors: {transfer_failed}"
                        f" | Processed: {success_count}"
                        f" | Failed: {failed_count}"
                    )

                    pending_results.clear()

                # Submit another task as soon as one finishes
                next_row = next(jobs_iterator, None)

                if next_row is not None:

                    new_future = executor.submit(
                        process_video,
                        next_row
                    )

                    active_futures[new_future] = True

        progress.close()

        # Transfer the final incomplete batch
        if pending_results:

            transferred, transfer_failed = transfer_batch(
                pending_results
            )

            transfer_count += transferred
            transfer_failed_count += transfer_failed

            pending_results.clear()


# =========================================================
# 7. Final report
# =========================================================

valid_negative = sum(
    1
    for path in DRIVE_NEGATIVE.glob("*.npz")
    if is_valid_npz(path)
)

print("\n" + "=" * 55)
print("NEGATIVE VIDEO PROCESSING REPORT")
print("=" * 55)

print(f"Total negative videos: {len(negative_df)}")
print(f"Processed successfully: {success_count}")
print(f"Processing failures: {failed_count}")
print(f"Transferred to Drive: {transfer_count}")
print(f"Transfer failures: {transfer_failed_count}")
print(f"Valid NPZ files in Drive: {valid_negative}")
print(f"Remaining videos: {len(negative_df) - valid_negative}")

print("=" * 55)

logging.info(
    f"Run completed | Success={success_count} "
    f"| Failed={failed_count} "
    f"| Transferred={transfer_count} "
    f"| Valid={valid_negative}"
)


train/negative/02085.mp4: reconstructing file:   0%|          |  0.00B / 14.5MB            

train/negative/02085.mp4: downloading bytes:           |  0.00B            

train/negative/02035.mp4: reconstructing file:   0%|          |  0.00B / 23.3MB            

train/negative/02035.mp4: downloading bytes:           |  0.00B            

train/negative/01623.mp4: reconstructing file:   0%|          |  0.00B / 9.59MB            

train/negative/01623.mp4: downloading bytes:           |  0.00B            

... (571 outputs left out)

train/negative/02029.mp4: downloading bytes:           |  0.00B            

train/negative/01320.mp4: reconstructing file:   0%|          |  0.00B / 20.0MB            

train/negative/01320.mp4: downloading bytes:           |  0.00B            


NEGATIVE VIDEO PROCESSING REPORT
Total negative videos: 750
Processed successfully: 299
Processing failures: 0
Transferred to Drive: 299
Transfer failures: 0
Valid NPZ files in Drive: 750
Remaining videos: 0


In [8]:

import pandas as pd

metadata_path = "/content/drive/MyDrive/Nexar_Project/Metadata/all_metadata.csv"

df = pd.read_csv(metadata_path)

print("Columns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nExisting NPZ files:")
import os

positive_dir = "/content/drive/MyDrive/Nexar_Project/Dataset_2FPS/positive"
negative_dir = "/content/drive/MyDrive/Nexar_Project/Dataset_2FPS/negative"

print("Positive:", len(os.listdir(positive_dir)))
print("Negative:", len(os.listdir(negative_dir)))

Columns:
['file_name', 'time_of_event', 'time_of_alert', 'light_conditions', 'weather', 'scene', 'time_to_accident', 'label']

First 5 rows:


,file_name,time_of_event,time_of_alert,light_conditions,weather,scene,time_to_accident,label
0,00822.mp4,19.500,18.633,Normal,Cloudy,Urban,NaN,1
1,00208.mp4,19.800,19.233,Normal,Cloudy,Sub-urban,NaN,1
2,00072.mp4,20.101,20.068,Normal,Clear,Urban,NaN,1
3,00128.mp4,19.267,18.333,Normal,Clear,Urban,NaN,1
4,00205.mp4,21.146,18.196,Normal,Clear,Highway,NaN,1



Data types:
file_name            object
time_of_event       float64
time_of_alert       float64
light_conditions     object
weather              object
scene                object
time_to_accident    float64
label                 int64
dtype: object

Existing NPZ files:
Positive: 750
Negative: 45


In [10]:
from pathlib import Path
import shutil
import os

project = Path("/content/drive/MyDrive/Nexar_Project")

positive = project / "Dataset_2FPS/positive"
negative = project / "Dataset_2FPS/negative"
staging = Path("/content/Nexar_Staging")

print("Positive NPZ files:", len(list(positive.glob("*.npz"))))
print("Negative NPZ files:", len(list(negative.glob("*.npz"))))

print("\nLocal staging files:")
print("Positive:", len(list((staging / "positive").glob("*.npz"))))
print("Negative:", len(list((staging / "negative").glob("*.npz"))))

print("\nColab local disk:")
print("Free space (GB):", round(shutil.disk_usage("/content").free / (1024**3), 2))

print("\nDrive mounted:", os.path.exists(project))

Positive NPZ files: 750
Negative NPZ files: 45

Local staging files:
Positive: 0
Negative: 0

Colab local disk:
Free space (GB): 107.17

Drive mounted: True


In [12]:
from pathlib import Path
import pandas as pd

project = Path("/content/drive/MyDrive/Nexar_Project")

df = pd.read_csv(project / "Metadata/all_metadata.csv")

positive_files = set(
    (project / "Dataset_2FPS/positive").glob("*.npz")
)

negative_files = set(
    (project / "Dataset_2FPS/negative").glob("*.npz")
)

expected_positive = set(
    Path(x).stem
    for x in df.loc[df["label"] == 1, "file_name"]
)

expected_negative = set(
    Path(x).stem
    for x in df.loc[df["label"] == 0, "file_name"]
)

actual_positive = {p.stem for p in positive_files}
actual_negative = {p.stem for p in negative_files}

print("Expected positive:", len(expected_positive))
print("Expected negative:", len(expected_negative))

print("Existing positive:", len(actual_positive))
print("Existing negative:", len(actual_negative))

print("Missing positive:", len(expected_positive - actual_positive))
print("Missing negative:", len(expected_negative - actual_negative))

print("Unexpected positive:", len(actual_positive - expected_positive))
print("Unexpected negative:", len(actual_negative - expected_negative))

Expected positive: 750
Expected negative: 750
Existing positive: 750
Existing negative: 85
Missing positive: 0
Missing negative: 665
Unexpected positive: 0
Unexpected negative: 0
